### Taxomic Analysis

*Preliminaries*

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from glob import glob

In [ ]:
BLAST_DIR = "/data/taxonomy/2024"  # path to BLAST result .tsv files
OUTPUT_DIR = "/data/reports/2024"

os.makedirs(OUTPUT_DIR, exist_ok=True)

blast_files = glob(os.path.join(BLAST_DIR, "*_blast.tsv"))

# Expected BLAST columns
cols = ["qseqid", "sseqid", "pident", "length", "qcov", "stitle"]
blast_dfs = []

# Parse each file and add sample label
for f in blast_files:
    sample = os.path.basename(f).replace("_blast.tsv","")
    df = pd.read_csv(f, sep="\t", names=cols)
    df["Sample"] = sample
    blast_dfs.append(df)

# Combine all into one dataframe
blast_all = pd.concat(blast_dfs, ignore_index = True)
blast_all.head()

*Extracting Species Names and Assigning Top Hits*

In [ ]:
# Use GenBank title as a proxy for taxon name
blast_all["Taxon"] = blast_all["stitle"].str.extract(r"^([^,]+)")

blast_top = (blast_all
             .sort_values(["Sample", "qseqid", "pident", "qcov"], ascending=[True, True, False, False])
             .drop_duplicates(subset=["Sample", "qseqid"]))

blast_top[["Sample", "qseqid", "Taxon"]].head()


*Constructing Abundance Table (Consensus Counts)*

In [ ]:
abundance = (
    blast_top.groupby(["Sample", "Taxon"])
    .size()
    .reset_index(name="Count")
)

# Pivot to wide-format abundance table
abundance_table = abundance.pivot(index="Taxon", columns="Sample", values="Count").fillna(0).astype(int)
abundance_table.to_csv(os.path.join(OUTPUT_DIR, "Taxonomic_Abundance_Table.csv"))
abundance_table.head()

*Visualize Taxonomic Composition (Barplot)*

In [ ]:
rel_abundance = abundance_table.div(abundance_table.sum(axis=0), axis=1)

# Plot stacked bar chart of top taxa
top_taxa = abundance_table.sum(axis=1).sort_values(ascending=False).head(10).index
plot_df = rel_abundance.loc[top_taxa].T

plt.figure(figsize=(10, 6))
plot_df.plot(kind='bar', stacked=True, colormap='tab10', figsize=(12, 6))
plt.ylabel("Relative Abundance")
plt.title("Taxonomic Composition per Sample")
plt.xticks(rotation=45, ha='right')
plt.legend(title="Taxon", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "taxonomic_composition_barplot.png"))
plt.show()

*Summary Table*

In [ ]:
summary = (
    blast_top.groupby("Sample")["Taxon"]
    .agg(["nunique"])
    .rename(columns={"nunique": "Unique_Taxa"})
    .assign(Consensus_Count=blast_top.groupby("Sample").size())
)

summary.to_csv(os.path.join(OUTPUT_DIR, "Sample_Summary_Report.csv"))
summary